In [2]:
# # For Data Analysis and Visualization
# !pip install pandas numpy matplotlib seaborn plotly missingno

# # For Machine Learning and AI
# !pip install scikit-learn tensorflow torch torchvision torchaudio

# # For Financial Data Analysis
# !pip install FinMind twstock

# # For Text Processing
# !pip install nltk

# # For Statistical Modeling
# !pip install scipy statsmodels

# # For Network Requests and Character Encoding
# !pip install requests chardet

# # For Technical Analysis on Financial Data
# !pip install C:\Users\archi\Python\Project\TA_Lib-0.4.24-cp37-cp37m-win_amd64.whl

# # Miscellaneous Utilities
# !pip install tqdm ast warnings datetime flit

In [3]:
# Standard libraries
from datetime import date, datetime, timedelta
import os
import time
import logging
import traceback
import shutil
import glob

# Data Analysis
import pandas as pd
import numpy as np

# Character Encoding
import chardet

# Technical Analysis
import talib

In [4]:
# whole
file_path = r"D:\Min\Python\Project\FA_Data\meta_data\stock_data_whole.csv"
stock_data_whole =pd.read_csv(file_path)

C:\Users\archi\AppData\Local\Temp\ipykernel_69168\1950031868.py:3: DtypeWarning: Columns (0,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  stock_data_whole =pd.read_csv(file_path)


## Save to UTF-8 CSV

In [5]:
print(stock_data_whole.head())
print(stock_data_whole.shape)
print(stock_data_whole['日期'].min(), stock_data_whole['日期'].max())
# print(stock_id_list.shape)

   證券代號 證券名稱      成交股數  成交筆數       成交金額    開盤價    最高價    最低價    收盤價 漲跌(+/-)  \
0  1101   台泥  16274945  3746  765303507  47.20  47.35  46.70  47.30       +   
1  1101   台泥  11662998  2840  550523184  47.15  47.55  46.90  46.90       -   
2  1101   台泥   6199830  2441  295121414  47.00  47.80  47.00  47.80       +   
3  1101   台泥  18284909  5533  891433797  48.20  49.50  48.05  49.50       +   
4  1101   台泥   9700134  5117  478372422  49.50  49.55  48.90  49.50     NaN   

   漲跌價差 最後揭示買價  最後揭示買量 最後揭示賣價  最後揭示賣量    本益比          日期  
0   0.1  47.20     1.0  47.30  1036.0  17.39  2014-04-07  
1   0.4  46.90   122.0  47.00     7.0  17.24  2014-04-08  
2   0.9  47.65    12.0  47.80   541.0  17.57  2014-04-09  
3   1.7  49.45     4.0  49.50    44.0  18.20  2014-04-10  
4   0.0  49.50    24.0  49.55   119.0  18.20  2014-04-11  
(2466511, 17)
2014-04-07 2024-10-29


In [6]:
# 確認證券代號欄位的數據類型
print(stock_data_whole['證券代號'].dtype)

# 確認是否包含 2330
print(stock_data_whole['證券代號'].unique())

# 如果證券代號為整數，將其轉為字串以避免匹配錯誤
stock_data_whole['證券代號'] = stock_data_whole['證券代號'].astype(str)

# 再次篩選出證券代號為 "2330" 的資料
data_2330 = stock_data_whole[stock_data_whole['證券代號'] == "2330"]

# 確認篩選出的資料是否有數據
if not data_2330.empty:
    min_date_2330 = data_2330['日期'].min()
    max_date_2330 = data_2330['日期'].max()
    print("證券代號2330的最小日期:", min_date_2330)
    print("證券代號2330的最大日期:", max_date_2330)
else:
    print("資料集中不包含證券代號 2330。")

object
['1101' '1101B' '1102' ... 9946 9955 9958]
證券代號2330的最小日期: 2014-04-07
證券代號2330的最大日期: 2024-10-29


## save indicators to CSV by stock_id

In [7]:
def calculate_and_store_indicators(df, stock_id=None, date=None, by_date=False):
    """
    Calculate technical indicators and save to files, organized by either stock_id or date
    
    Parameters:
    df (DataFrame): Input data
    stock_id (str): Stock ID if organizing by stock
    date (str): Date if organizing by date  
    by_date (bool): Whether to organize by date
    """
    try:
        # 數據預處理
        cols_to_check = ['收盤價', '最高價', '最低價']
        for col in cols_to_check:
            df[col] = pd.to_numeric(df[col].replace('--', np.nan), errors='coerce')
            
        df.sort_values(by='日期', inplace=True)

        # 檢查數據有效性
        if df[cols_to_check].isnull().all().any():
            print(f"No valid price data for {'date '+date if by_date else 'stock ID '+stock_id}")
            return
            
        # 計算技術指標
        df['SMA30'] = talib.SMA(df['收盤價'], timeperiod=30)
        df['DEMA30'] = talib.DEMA(df['收盤價'], timeperiod=30)
        df['EMA30'] = talib.EMA(df['收盤價'], timeperiod=30)
        df['RSI'] = talib.RSI(df['收盤價'], timeperiod=14)
        df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(df['收盤價'], fastperiod=12, slowperiod=26, signalperiod=9)
        df['slowk'], df['slowd'] = talib.STOCH(df['最高價'], df['最低價'], df['收盤價'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)
        df['TSF'] = talib.TSF(df['收盤價'], timeperiod=14)
        df['middleband'], _, _ = talib.BBANDS(df['收盤價'], timeperiod=30, nbdevup=2, nbdevdn=2, matype=0)
        df['SAR'] = talib.SAR(df['最高價'], df['最低價'], acceleration=0.02, maximum=0.2)

        # 儲存路徑設定
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        
        # 1. 個別檔案儲存(依股票代號或日期)
        if by_date:
            individual_path = f"{base_path}\\technical_analysis\\by_date\\{date}_indicators.csv"
            df['日期'] = date  # 確保日期欄位統一
        else:
            individual_path = f"{base_path}\\technical_analysis\\{stock_id}_indicators.csv"
        
        # 建立目錄(如果不存在)
        os.makedirs(os.path.dirname(individual_path), exist_ok=True)
        
        # 儲存個別檔案
        df.to_csv(individual_path, index=False, encoding='utf-8-sig')
        
        # 2. 更新全部股票數據
        all_stocks_path = f"{base_path}\\meta_data\\all_stocks_data.csv"
        
        if os.path.exists(all_stocks_path):
            try:
                all_stocks_df = pd.read_csv(all_stocks_path, low_memory=False)  # 加入 low_memory=False
                if by_date:
                    # 如果是依日期，移除該日期的舊數據
                    all_stocks_df = all_stocks_df[all_stocks_df['日期'] != date]
                else:
                    # 如果是依股票，移除該股票的舊數據
                    all_stocks_df = all_stocks_df[all_stocks_df['證券代號'] != stock_id]
                # 合併新數據
                all_stocks_df = pd.concat([all_stocks_df, df], ignore_index=True)
            except Exception as e:
                print(f"Error reading all_stocks_data.csv: {e}")
                all_stocks_df = df.copy()
            
        # 儲存前先檢查今天是否已經備份
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        all_stocks_path = f"{base_path}\\meta_data\\all_stocks_data.csv"
        today = datetime.now().strftime('%Y%m%d')
        backup_dir = f"{base_path}\\meta_data\\backup"
        today_backup = f"{backup_dir}\\all_stocks_data_{today}.csv"
        
        # 如果今天還沒備份且原始檔案存在，則進行備份
        if os.path.exists(all_stocks_path) and not os.path.exists(today_backup):
            os.makedirs(backup_dir, exist_ok=True)
            shutil.copy2(all_stocks_path, today_backup)
            # 清理舊備份，只保留最近7天的
            cleanup_old_backups(backup_dir, keep_days=7)
            
        all_stocks_df.to_csv(all_stocks_path, index=False, encoding='utf-8-sig')
        
        print(f"Indicators saved successfully for {'date '+date if by_date else 'stock ID '+stock_id}")

    except Exception as e:
        print(f"Failed to process {'date '+date if by_date else 'stock ID '+stock_id} due to: {e}")
        print(traceback.format_exc())

In [8]:
def process_stock_data_batch():
    """批次處理股票資料"""
    try:
        base_path = "D:\\Min\\Python\\Project\\FA_Data"
        # 讀取原始資料
        stock_data_whole = pd.read_csv(f"{base_path}\\meta_data\\stock_data_whole.csv", low_memory=False)
        
        # 建立空的 DataFrame 來存儲所有處理後的資料
        all_processed_data = []
        
        # 按股票代號分組處理
        grouped = stock_data_whole.groupby('證券代號')
        total_stocks = len(grouped)
        
        for idx, (stock_id, group_df) in enumerate(grouped, 1):
            try:
                # 數據預處理
                df = group_df.copy()
                
                # 處理價格欄位 - 移除千分位逗號並轉換為float
                price_columns = ['收盤價', '最高價', '最低價']
                for col in price_columns:
                    # 先將 '--' 替換為 NaN，然後移除逗號
                    df[col] = df[col].replace('--', np.nan)
                    df[col] = df[col].astype(str).str.replace(',', '')
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                
                df.sort_values(by='日期', inplace=True)
                
                # 移除所有價格都是 NaN 的列
                df = df.dropna(subset=price_columns, how='all')
                
                # 檢查是否還有足夠的數據
                if len(df) < 30:  # 至少需要30天的數據來計算技術指標
                    print(f"Insufficient data for stock ID {stock_id}")
                    continue
                
                # 計算技術指標
                df['SMA30'] = talib.SMA(df['收盤價'], timeperiod=30)
                df['DEMA30'] = talib.DEMA(df['收盤價'], timeperiod=30)
                df['EMA30'] = talib.EMA(df['收盤價'], timeperiod=30)
                df['RSI'] = talib.RSI(df['收盤價'], timeperiod=14)
                df['MACD'], df['MACD_signal'], df['MACD_hist'] = talib.MACD(
                    df['收盤價'], fastperiod=12, slowperiod=26, signalperiod=9
                )
                
                # 只在有完整的高低價數據時計算STOCH
                if not df[['最高價', '最低價', '收盤價']].isnull().any().any():
                    df['slowk'], df['slowd'] = talib.STOCH(
                        df['最高價'], df['最低價'], df['收盤價'],
                        fastk_period=5, slowk_period=3, slowk_matype=0,
                        slowd_period=3, slowd_matype=0
                    )
                
                df['TSF'] = talib.TSF(df['收盤價'], timeperiod=14)
                df['middleband'], _, _ = talib.BBANDS(
                    df['收盤價'], timeperiod=30, nbdevup=2, nbdevdn=2, matype=0
                )
                
                # 只在有完整的高低價數據時計算SAR
                if not df[['最高價', '最低價']].isnull().any().any():
                    df['SAR'] = talib.SAR(df['最高價'], df['最低價'], acceleration=0.02, maximum=0.2)
                
                # 儲存個別股票的指標檔案
                individual_path = f"{base_path}\\technical_analysis\\{stock_id}_indicators.csv"
                os.makedirs(os.path.dirname(individual_path), exist_ok=True)
                df.to_csv(individual_path, index=False, encoding='utf-8-sig')
                
                # 將處理後的資料加入列表
                all_processed_data.append(df)
                
                print(f"Processed {idx}/{total_stocks} - Stock ID: {stock_id}")
                
            except Exception as e:
                print(f"Error processing stock {stock_id}: {str(e)}")
                continue
        
        if all_processed_data:
            # 合併所有處理後的資料
            all_stocks_df = pd.concat(all_processed_data, ignore_index=True)
            
            # 儲存前先備份
            today = datetime.now().strftime('%Y%m%d')
            backup_dir = f"{base_path}\\meta_data\\backup"
            today_backup = f"{backup_dir}\\all_stocks_data_{today}.csv"
            
            if not os.path.exists(today_backup):
                os.makedirs(backup_dir, exist_ok=True)
                all_stocks_df.to_csv(today_backup, index=False, encoding='utf-8-sig')
                cleanup_old_backups(backup_dir, keep_days=7)
            
            # 儲存完整的處理後資料
            all_stocks_df.to_csv(f"{base_path}\\meta_data\\all_stocks_data.csv", index=False, encoding='utf-8-sig')
            
            return True
        else:
            print("No data was successfully processed")
            return False
        
    except Exception as e:
        print(f"Error in batch processing: {str(e)}")
        return False

In [9]:
def cleanup_old_backups(backup_dir, keep_days=7):
    """清理舊的備份檔案，只保留最近N天的"""
    try:
        # 取得所有備份檔案
        backup_files = glob.glob(os.path.join(backup_dir, 'all_stocks_data_*.csv'))
        # 依檔案修改時間排序
        backup_files.sort(key=os.path.getmtime)
        
        # 如果備份檔案數量超過要保留的天數，刪除最舊的
        if len(backup_files) > keep_days:
            for old_file in backup_files[:-keep_days]:
                try:
                    os.remove(old_file)
                    print(f"Removed old backup: {old_file}")
                except Exception as e:
                    print(f"Failed to remove {old_file}: {e}")
    except Exception as e:
        print(f"Error during backup cleanup: {e}")

In [10]:
def cleanup_stock_data():
    """
    清理和標準化股票數據
    """
    base_path = "D:\\Min\\Python\\Project\\FA_Data"
    all_stocks_path = f"{base_path}\\meta_data\\all_stocks_data.csv"
    
    try:
        # 讀取全部股票數據
        df = pd.read_csv(all_stocks_path, low_memory=False, parse_dates=['日期'])
        
        # 清理工作
        df['證券代號'] = df['證券代號'].astype(str)
        df['是普通股'] = df['證券代號'].str.isdigit()
        df = df[df['是普通股']].copy()
        df = df.drop(['是普通股'], axis=1)
        
        # 標準化公司名稱
        df['標準化公司名稱'] = df['證券名稱'].str.replace(r'-.*$', '', regex=True)
        
        # 處理名稱變更
        name_changes = df.groupby('證券代號')['證券名稱'].nunique() > 1
        if name_changes.any():
            print("\n發現公司名稱變更:")
            for code in name_changes[name_changes].index:
                changes = df[df['證券代號'] == code][['日期', '證券名稱']].drop_duplicates()
                latest_name = changes.sort_values('日期').iloc[-1]['證券名稱']
                df.loc[df['證券代號'] == code, '標準化公司名稱'] = latest_name
                print(f"股票代號 {code}: 最新名稱為 {latest_name}")
        
        # 儲存前先檢查今天是否已經備份
        today = datetime.now().strftime('%Y%m%d')
        backup_dir = f"{base_path}\\meta_data\\backup"
        today_backup = f"{backup_dir}\\all_stocks_data_{today}_cleaned.csv"
        
        # 如果今天還沒備份且原始檔案存在，則進行備份
        if os.path.exists(all_stocks_path) and not os.path.exists(today_backup):
            os.makedirs(backup_dir, exist_ok=True)
            shutil.copy2(all_stocks_path, today_backup)
            # 清理舊備份，只保留最近7天的
            cleanup_old_backups(backup_dir, keep_days=7)
        
        # 重新儲存清理後的數據
        df.to_csv(all_stocks_path, index=False, encoding='utf-8-sig')
        
        # 打印數據統計
        print("\n數據統計:")
        print(f"總行數: {len(df)}")
        print(f"唯一證券代號數: {df['證券代號'].nunique()}")
        print(f"時間範圍: 從 {df['日期'].min()} 到 {df['日期'].max()}")
        
        return True
        
    except Exception as e:
        print(f"數據清理過程發生錯誤: {e}")
        print(traceback.format_exc())
        return False

In [11]:
def main():
    """
    主程序
    """
    try:
        # 設置日誌
        log_file = "stock_data_processing.log"
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file, encoding='utf-8'),
                logging.StreamHandler()
            ]
        )
        
        # 批次處理資料
        logging.info("Starting batch processing...")
        if process_stock_data_batch():
            logging.info("Batch processing completed successfully")
        else:
            logging.error("Batch processing failed")
            
    except Exception as e:
        logging.error(f"Error in main process: {e}")
        logging.error(traceback.format_exc())

In [12]:
if __name__ == "__main__":
    main()

2024-10-29 15:01:11,041 - INFO - Starting batch processing...


Processed 1/1148 - Stock ID: 1101
Processed 2/1148 - Stock ID: 1101B
Processed 3/1148 - Stock ID: 1102
Processed 4/1148 - Stock ID: 1103
Processed 5/1148 - Stock ID: 1104
Processed 6/1148 - Stock ID: 1108
Processed 7/1148 - Stock ID: 1109
Processed 8/1148 - Stock ID: 1110
Processed 9/1148 - Stock ID: 1201
Processed 10/1148 - Stock ID: 1203
Processed 11/1148 - Stock ID: 1210
Processed 12/1148 - Stock ID: 1213
Processed 13/1148 - Stock ID: 1215
Processed 14/1148 - Stock ID: 1216
Processed 15/1148 - Stock ID: 1217
Processed 16/1148 - Stock ID: 1218
Processed 17/1148 - Stock ID: 1219
Processed 18/1148 - Stock ID: 1220
Processed 19/1148 - Stock ID: 1225
Processed 20/1148 - Stock ID: 1227
Processed 21/1148 - Stock ID: 1229
Processed 22/1148 - Stock ID: 1231
Processed 23/1148 - Stock ID: 1232
Processed 24/1148 - Stock ID: 1233
Processed 25/1148 - Stock ID: 1234
Processed 26/1148 - Stock ID: 1235
Processed 27/1148 - Stock ID: 1236
Processed 28/1148 - Stock ID: 1256
Processed 29/1148 - Stock ID

C:\Users\archi\AppData\Local\Temp\ipykernel_69168\475941706.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace('--', np.nan)
C:\Users\archi\AppData\Local\Temp\ipykernel_69168\475941706.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace('--', np.nan)
C:\Users\archi\AppData\Local\Temp\ipykernel_69168\475941706.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_o

Processed 1095/1148 - Stock ID: 911608
Processed 1096/1148 - Stock ID: 911609
Processed 1097/1148 - Stock ID: 911611
Processed 1098/1148 - Stock ID: 911612
Processed 1099/1148 - Stock ID: 911616
Processed 1100/1148 - Stock ID: 911619
Processed 1101/1148 - Stock ID: 911622
Processed 1102/1148 - Stock ID: 911626
Processed 1103/1148 - Stock ID: 911868
Processed 1104/1148 - Stock ID: 912000
Processed 1105/1148 - Stock ID: 912398
Processed 1106/1148 - Stock ID: 9136
Processed 1107/1148 - Stock ID: 913889
Processed 1108/1148 - Stock ID: 9157
Processed 1109/1148 - Stock ID: 9188
Processed 1110/1148 - Stock ID: 9802
Processed 1111/1148 - Stock ID: 9902
Processed 1112/1148 - Stock ID: 9904
Processed 1113/1148 - Stock ID: 9905
Processed 1114/1148 - Stock ID: 9906
Processed 1115/1148 - Stock ID: 9907
Processed 1116/1148 - Stock ID: 9908
Processed 1117/1148 - Stock ID: 9910
Processed 1118/1148 - Stock ID: 9911
Processed 1119/1148 - Stock ID: 9912
Processed 1120/1148 - Stock ID: 9914
Processed 1121

2024-10-29 15:03:39,939 - INFO - Batch processing completed successfully


Processed 924/1148 - Stock ID: 6456
Processed 925/1148 - Stock ID: 6464
Processed 926/1148 - Stock ID: 6472
Processed 927/1148 - Stock ID: 6477
Processed 928/1148 - Stock ID: 6491
Processed 929/1148 - Stock ID: 6504
Processed 930/1148 - Stock ID: 6505
Processed 931/1148 - Stock ID: 6515
Processed 932/1148 - Stock ID: 6525
Processed 933/1148 - Stock ID: 6526
Processed 934/1148 - Stock ID: 6531
Processed 935/1148 - Stock ID: 6533
Processed 936/1148 - Stock ID: 6534
Processed 937/1148 - Stock ID: 6541
Processed 938/1148 - Stock ID: 6550
Processed 939/1148 - Stock ID: 6552
Processed 940/1148 - Stock ID: 6558
Processed 941/1148 - Stock ID: 6573
Processed 942/1148 - Stock ID: 6579
Processed 943/1148 - Stock ID: 6581
Processed 944/1148 - Stock ID: 6582
Processed 945/1148 - Stock ID: 6585
Processed 946/1148 - Stock ID: 6591
Processed 947/1148 - Stock ID: 6592
Processed 948/1148 - Stock ID: 6592A
Processed 949/1148 - Stock ID: 6592B
Processed 950/1148 - Stock ID: 6598
Processed 951/1148 - Stock

Processed 1147/1148 - Stock ID: 9955
Processed 1148/1148 - Stock ID: 9958


2024-10-27 14:39:42,337 - INFO - Batch processing completed successfully


In [13]:
file_path_test = r"D:\Min\Python\Project\FA_Data\technical_analysis\2330_indicators.csv"
test =pd.read_csv(file_path_test)

test

,證券代號,證券名稱,成交股數,成交筆數,成交金額,開盤價,最高價,最低價,收盤價,漲跌(+/-),...,EMA30,RSI,MACD,MACD_signal,MACD_hist,slowk,slowd,TSF,middleband,SAR
0,2330,台積電,45254843,11316,5305535631,117.50,118.0,116.5,117.5,-,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2330,台積電,43894168,9396,5173993549,117.00,119.5,117.0,119.0,+,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116.500000
2,2330,台積電,29648114,8284,3548296766,119.50,120.0,119.0,119.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116.560000
3,2330,台積電,27215773,7229,3243663187,119.00,119.5,118.5,119.5,+,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116.697600
4,2330,台積電,34847792,7639,4153152840,118.50,120.0,118.0,120.0,+,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,116.829696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2579,2330,台積電,32895421,40549,34912308143,"1,060.00",1070.0,1055.0,1060.0,-,...,1010.501072,60.763201,30.714842,26.892117,3.822725,61.904762,65.728716,1096.582418,1102.967067,1032.772000
2580,2330,台積電,40791484,42113,43477824029,"1,070.00",1075.0,1055.0,1060.0,NaN,...,1013.694551,60.763201,29.450019,27.403697,2.046322,39.417989,58.377425,1091.362637,1106.537164,1034.116560
2581,2330,台積電,23347890,25884,24867260319,"1,065.00",1070.0,1060.0,1065.0,+,...,1017.004580,61.748523,28.522309,27.627420,0.894890,26.322751,42.548501,1085.549451,1108.156970,1035.434229
2582,2330,台積電,41665065,55098,44216316045,"1,075.00",1080.0,1050.0,1050.0,-,...,1019.133317,57.114713,26.273849,27.356706,-1.082856,12.037037,25.925926,1077.472527,1107.470685,1036.725544
